# SIEM Log Analysis — Llama-3-8B Fine-tuning (Fixed Notebook)

Is notebook mein saari pichli mistakes fix ki gayi hain:
1. **GPU memory leak fix** — model ko baar baar reload karne se pehle purana model explicitly free kiya jata hai (RTX 4060 jaisi 7-8GB GPU ke liye zaroori)
2. **Real-time inference mein ground truth nahi di jati** — sirf prompt banta hai, response kabhi input mein nahi jata
3. **Numerical fields (risk_score, confidence) ka proper evaluation** — sirf visual check nahi, MAE (Mean Absolute Error) metric se measure kiya jata hai
4. Config (logging/eval/save) hamesha `trainer.train()` se **pehle** set ki jati hai

## Step 1: Dataset load karna

In [ ]:
!pip install -q datasets scikit-learn pandas

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("darkknight25/Advanced_SIEM_Dataset", split="train")
print("Dataset loaded successfully!")
print(dataset)

df = dataset.to_pandas()

# advanced_metadata ek dict column hai — usme se risk_score aur confidence nikalna
df["risk_score"] = df["advanced_metadata"].apply(lambda x: x["risk_score"])
df["confidence"] = df["advanced_metadata"].apply(lambda x: x["confidence"])

## Step 2: Basic preprocessing

In [ ]:
import numpy as np

# dst_ip mein missing/N-A values ko placeholder se replace karna
df["dst_ip"] = (
    df["dst_ip"]
    .astype(str)
    .str.strip()
    .replace({"N/A": "0.0.0.0", "n/a": "0.0.0.0", "NA": "0.0.0.0", "": "0.0.0.0"})
)

# timestamp ko datetime mein convert karna (temporal analysis / sorting ke liye)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df.shape)
df.head(2)

## Step 3: Sirf teen sources filter karna

Poore 100k rows ke bajaye sirf `Suricata`, `Elastic SIEM`, aur `Wazuh` ka data rakhte hain —
taake training free-tier GPU (Colab T4 / local 7-8GB card) pe reasonable time mein ho sake.

In [ ]:
selected_sources = ["Suricata v6.0.10", "Elastic SIEM v8.9.0", "Wazuh v4.5.0"]

df_filtered = df[df["source"].isin(selected_sources)].reset_index(drop=True)

print("Filtered rows:", df_filtered.shape[0])
print(df_filtered["source"].value_counts())

df = df_filtered

## Step 4: Instruction / Response format banana

**Fix:** `Process ID` ko float (`8141.0`) ki jagah clean integer (`8141`) banaya gaya hai.
Sparse fields (jo kisi row mein NaN hain) automatically skip ho jaati hain.

In [ ]:
def is_valid(val):
    if val is None:
        return False
    if isinstance(val, float) and np.isnan(val):
        return False
    if isinstance(val, str) and val.strip().lower() in ["", "n/a", "na", "none", "null", "nan"]:
        return False
    return True

def flatten_dict(val, keep_keys=None):
    """Dict ko readable 'key=value; key=value' string mein convert karta hai"""
    if not isinstance(val, dict) or len(val) == 0:
        return None
    if keep_keys:
        val = {k: v for k, v in val.items() if k in keep_keys}
    if not val:
        return None
    return "; ".join(f"{k}={v}" for k, v in val.items())

# behavioral_analytics aur advanced_metadata dono dict columns hain
df["_behavioral_parsed"] = df["behavioral_analytics"].apply(flatten_dict)
df["_metadata_parsed"] = df["advanced_metadata"].apply(
    lambda x: flatten_dict(x, keep_keys=["geo_location", "user_agent"])
)

def build_instruction(row):
    pid = row.get("process_id")
    pid_str = str(int(pid)) if is_valid(pid) else None

    input_fields = {
        "Timestamp": row.get("timestamp"),
        "Source": row.get("source"),
        "Raw Log": row.get("raw_log"),
        "User": row.get("user"),
        "Action": row.get("action"),
        "Object": row.get("object"),
        "Process ID": pid_str,
        "Parent Process": row.get("parent_process"),
        "Device Type": row.get("device_type"),
        "Device ID": row.get("device_id"),
        "Firmware Version": row.get("firmware_version"),
        "Source IP": row.get("src_ip"),
        "Destination IP": row.get("dst_ip"),
        "Source Port": row.get("src_port"),
        "Destination Port": row.get("dst_port"),
        "Protocol": row.get("protocol"),
        "Method": row.get("method"),
        "Bytes Transferred": row.get("bytes"),
        "Duration": row.get("duration"),
        "MAC Address": row.get("mac_address"),
        "Cloud Service": row.get("cloud_service"),
        "Resource ID": row.get("resource_id"),
        "Geo Location": row.get("_metadata_parsed"),
        "Additional Info": row.get("additional_info"),
        "Behavioral Analytics": row.get("_behavioral_parsed"),
    }
    input_lines = [f"{k}: {v}" for k, v in input_fields.items() if is_valid(v)]
    instruction = (
        "Analyze the following SIEM event log and classify its event type, "
        "severity, category, risk score, and provide a brief description.\n\n"
        + "\n".join(input_lines)
    )

    output_fields = {
        "Event Type": row.get("event_type"),
        "Severity": row.get("severity"),
        "Category": row.get("category"),
        "Alert Type": row.get("alert_type"),
        "Signature ID": row.get("signature_id"),
        "Risk Score": row.get("risk_score"),
        "Confidence": row.get("confidence"),
        "Description": row.get("description"),
    }
    output_lines = [f"{k}: {v}" for k, v in output_fields.items() if is_valid(v)]
    response = "\n".join(output_lines)
    return instruction, response

results = df.apply(build_instruction, axis=1)
df["instruction"] = results.apply(lambda x: x[0])
df["response"] = results.apply(lambda x: x[1])

print(df["instruction"].iloc[0])
print("---")
print(df["response"].iloc[0])

## Step 5: Sanity check — koi row miss to nahi hui

In [ ]:
print(df.shape)
print(len(df["instruction"]))
print(len(df["response"]))
print(df["instruction"].isna().sum())
print(df["response"].isna().sum())

## Step 6: HuggingFace Dataset banana + train/validation split

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(df[["instruction", "response"]].reset_index(drop=True))

split_dataset = hf_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

## Step 7: Base model + tokenizer load karna (training ke liye)

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # agar aapki GPU 8GB se kam hai to 1024 use karein

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

## Step 8: Llama-3 chat template apply karna

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")

def to_text(row):
    conversation = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    row["text"] = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )
    return row

train_dataset = train_dataset.map(to_text)
eval_dataset = eval_dataset.map(to_text)

print(train_dataset[0]["text"][:600])

## Step 9: LoRA adapters lagana

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Step 10: Trainer setup

**Fix:** `save_strategy`/`save_steps`/`eval_strategy` sab **shuru se hi** `TrainingArguments` mein
set hain — pehle jaisa "train() call karke baad mein settings badalna" wala order-mistake yahan nahi hai.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 25,
        eval_strategy = "steps",
        eval_steps = 200,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

## Step 11: Training chalana

In [ ]:
trainer_stats = trainer.train()

## Step 12: Loss curve dekhna

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [x["loss"] for x in history if "loss" in x]
eval_loss = [x["eval_loss"] for x in history if "eval_loss" in x]

plt.plot(train_loss, label="Train Loss")
if eval_loss:
    plt.plot(eval_loss, label="Eval Loss")
plt.xlabel("Logging step")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Progress")
plt.show()

## Step 13: Inference test — controlled test (ground truth ke sath compare)

Ye sirf **validation ke liye** hai — eval_dataset se ek row uthate hain jismein hum
already jaante hain sahi jawab (`response`) kya hai, taake model ka output compare kar sakein.

In [ ]:
FastLanguageModel.for_inference(model)

def analyze_log_text(prompt_text, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True,
        temperature=0.3, do_sample=True,
    )
    result = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
    return result.strip()

test_prompt = eval_dataset[0]["instruction"]
print("PROMPT:\n", test_prompt)
print("\nMODEL OUTPUT:\n", analyze_log_text(test_prompt))
print("\nACTUAL ANSWER:\n", eval_dataset[0]["response"])

## Step 14: Numerical fields ka proper evaluation (Risk Score / Confidence)

**Kyun zaroori hai:** LLM text-generation model hai — categorical fields (event_type, severity)
mein achha perform karta hai, lekin continuous numbers (risk_score, confidence) sirf
approximate/pattern-match hoti hain, exact calculation nahi. Isliye sirf ek example dekh kar
"sahi lag raha hai" kehna kaafi nahi — Mean Absolute Error (MAE) nikalna chahiye.

In [ ]:
import re
import numpy as np

def extract_number(text, field_name):
    pattern = rf"{field_name}:\s*([\d.]+)"
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None

n_samples = min(50, len(eval_dataset))   # quick test ke liye 50 samples
errors_risk, errors_conf = [], []
missing_risk, missing_conf = 0, 0

for i in range(n_samples):
    prompt = eval_dataset[i]["instruction"]
    actual = eval_dataset[i]["response"]
    predicted = analyze_log_text(prompt)

    actual_risk = extract_number(actual, "Risk Score")
    pred_risk = extract_number(predicted, "Risk Score")
    actual_conf = extract_number(actual, "Confidence")
    pred_conf = extract_number(predicted, "Confidence")

    if actual_risk is not None and pred_risk is not None:
        errors_risk.append(abs(actual_risk - pred_risk))
    elif actual_risk is not None:
        missing_risk += 1

    if actual_conf is not None and pred_conf is not None:
        errors_conf.append(abs(actual_conf - pred_conf))
    elif actual_conf is not None:
        missing_conf += 1

print(f"Samples evaluated: {n_samples}")
print(f"Risk Score  -> MAE: {np.mean(errors_risk):.2f} | matched: {len(errors_risk)} | model missed field: {missing_risk}")
print(f"Confidence  -> MAE: {np.mean(errors_conf):.4f} | matched: {len(errors_conf)} | model missed field: {missing_conf}")

**Interpretation:**
- `Risk Score` MAE ~2-5 → model ne kaafi achi tarah numeric pattern seekh liya hai
- `Risk Score` MAE ~15-20+ → model sirf "guess" kar raha hai, iska matlab risk_score/confidence ko
  LLM se hataa kar ek separate simple regression model (StandardScaler + regressor) se calculate
  karna zyada reliable hoga, aur LLM sirf categorical fields (event_type, severity, description)
  ke liye use karein.

## Step 15: Model save karna

In [ ]:
model.save_pretrained("siem_lora_model")
tokenizer.save_pretrained("siem_lora_model")
print("Saved to siem_lora_model/")

## Step 16: Model reload karna (GPU memory-safe tareeqe se)

**FIX (yahi asal mistake thi):** Pehle training wala `model`/`tokenizer` GPU se explicitly free
karte hain, phir dobara load karte hain. Isse chhoti GPU (jaise 7-8GB) pe multiple model copies
ka load hoke CPU-offload hona aur `cuda vs cpu device mismatch` error aana band ho jayega.

Agar possible ho, **kernel/runtime restart karke fresh session** mein ye cell chalayein — sabse safe tareeqa.

In [ ]:
import gc
import torch

# Purana model/tokenizer GPU se hatana (agar memory mein hai)
if "model" in dir():
    del model
if "tokenizer" in dir():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel

max_seq_length = 1024   # chhoti GPU (jaise RTX 4060 8GB) ke liye 1024 rakhein

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "siem_lora_model",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("Model reloaded (memory-safe) and ready for inference.")

## Step 17: Real-time SIEM JSON ko training-format prompt mein convert karna

**FIX (dusri asal mistake):** Real-time inference mein hum **kabhi ground truth/answer nahi dete**
— sirf raw JSON se prompt banate hain, jaisa production mein hota (jahan sahi jawab pehle se
pata nahi hota, isiliye to model se pucha ja raha hai).

In [ ]:
def prepare_siem_prompt(event):
    source = event.get("_source", event)

    predecoder = source.get("predecoder", {})
    agent = source.get("agent", {})
    rule = source.get("rule", {})
    decoder = source.get("decoder", {})

    timestamp = source.get("timestamp") or predecoder.get("timestamp", "")
    program = predecoder.get("program_name", "")

    agent_name = agent.get("name", "")
    agent_ip = agent.get("ip", "")

    rule_id = rule.get("id", "")
    rule_level = rule.get("level", "")
    rule_description = rule.get("description", "")
    fired_times = rule.get("firedtimes", "")

    groups = rule.get("groups", [])
    if isinstance(groups, list):
        groups = ", ".join(groups)

    full_log = source.get("full_log", "")
    decoder_name = decoder.get("name", "")

    prompt = (
        "Analyze the following SIEM event log and classify its event type, severity, "
        "category, risk score, and provide a brief description.\n\n"
        f"Timestamp: {timestamp}\n"
        f"Source: {program}\n"
        f"Raw Log: {full_log}\n"
        f"User: {agent_name}\n"
        f"Action: {rule_description}\n"
        f"Source IP: {agent_ip}\n"
        f"Method: {decoder_name}\n"
        f"Additional Info: Rule ID {rule_id}, Rule Level {rule_level}, "
        f"firedtimes={fired_times}, groups={groups}\n"
    )
    return prompt


def analyze_real_time_log(event):
    """Real-time event ke liye — sirf prompt banata hai, koi ground truth involve nahi"""
    prompt = prepare_siem_prompt(event)

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            use_cache=True,
            do_sample=False,          # deterministic output real-time classification ke liye behtar
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return result.strip()

## Step 18: Real-time Wazuh JSON pe test karna

In [ ]:
real_time_log = {
    "_index": "detact-edr-alerts-4.x-2026.08.21",
    "_id": "1tSRIqABkfh5gt1cbTVG",
    "_source": {
        "predecoder": {
            "program_name": "systemd",
            "timestamp": "2026-08-21T04:25:47.433581+00:00"
        },
        "agent": {"ip": "192.168.0.189", "name": "huzaifa", "id": "065"},
        "rule": {
            "firedtimes": 294,
            "level": 5,
            "description": "Systemd: Service exited due to a failure.",
            "groups": ["local", "systemd"],
            "id": "40704"
        },
        "location": "/var/log/syslog",
        "decoder": {"name": "systemd"},
        "full_log": "2026-08-21T04:25:47.433581+00:00 ubuntu24 systemd[1]: adsys-boot.service: Main process exited, code=exited, status=1/FAILURE",
        "timestamp": "2026-08-21T04:25:48.388+0000"
    }
}

prediction = analyze_real_time_log(real_time_log)
print("MODEL PREDICTION:")
print(prediction)